# Brasil — Fire — Collection 5

Pipeline completo: mapas → área stats → gráficos → composição.
Dados **Fire — Collection 5** do [MapBiomas](https://mapbiomas.org/) via [Google Earth Engine](https://earthengine.google.com/).


## Territórios disponíveis

| Grupo | ID | Territórios | Qtd |
|---|---:|---:|---:|
| Biomes | `biomes` | bioma_amazonia, bioma_caatinga, bioma_cerrado … | 7 |
| Country | `country` | pais_brasil, pais_brasil_biomes | 2 |
| Regions | `regions` | regiao_bap, regiao_bap_planalto, regiao_centro_oeste … | 9 |
| States | `states` | uf_acre, uf_alagoas, uf_amapa … | 27 |


## Catálogo de produtos

| # | Produto | Descrição | Viz |
|---:|---|---|---|
| 1 | `accumulated_burned` | Área queimada acumulada | fire |
| 2 | `accumulated_burned_coverage_nivel0` | Área queimada por classe de cobertura e uso da terra - Acumulada - Nível 0 (Natural/Antrópico) | coverage_nivel0 |
| 3 | `accumulated_burned_coverage_nivel1` | Área queimada por classe de cobertura e uso da terra - Acumulada - Nível 1 | coverage_nivel1 |
| 4 | `accumulated_burned_coverage_nivel1_1` | Área queimada por classe de cobertura e uso da terra - Acumulada - Nível 1.1 | coverage_nivel1_1 |
| 5 | `accumulated_burned_coverage_nivel2` | Área queimada por classe de cobertura e uso da terra - Acumulada - Nível 2 | coverage_nivel2 |
| 6 | `accumulated_burned_coverage_nivel3` | Área queimada por classe de cobertura e uso da terra - Acumulada - Nível 3 | coverage_nivel3 |
| 7 | `accumulated_burned_coverage_nivel4` | Área queimada por classe de cobertura e uso da terra - Acumulada - Nível 4 | coverage_nivel4 |
| 8 | `annual_burned` | Área queimada anual | fire |
| 9 | `annual_burned_coverage_nivel0` | Área queimada por classe de cobertura e uso da terra - Anual - Nível 0 (Natural/Antrópico) | coverage_nivel0 |
| 10 | `annual_burned_coverage_nivel1` | Área queimada por classe de cobertura e uso da terra - Anual - Nível 1 | coverage_nivel1 |
| 11 | `annual_burned_coverage_nivel1_1` | Área queimada por classe de cobertura e uso da terra - Anual - Nível 1.1 | coverage_nivel1_1 |
| 12 | `annual_burned_coverage_nivel2` | Área queimada por classe de cobertura e uso da terra - Anual - Nível 2 | coverage_nivel2 |
| 13 | `annual_burned_coverage_nivel3` | Área queimada por classe de cobertura e uso da terra - Anual - Nível 3 | coverage_nivel3 |
| 14 | `annual_burned_coverage_nivel4` | Área queimada por classe de cobertura e uso da terra - Anual - Nível 4 | coverage_nivel4 |
| 15 | `fire_frequency` | Frequência da área queimada | fire_col5_frequency |
| 16 | `fire_return_interval` | Intervalo de retorno do fogo | fire_col5_return_interval |
| 17 | `mean_fire_return_interval` | Intervalo médio de retorno do fogo | fire_col5_return_interval |
| 18 | `monthly_burned` | Distribuição mensal da área queimada | fire_col5_monthly |
| 19 | `nbr_min` | Mosaico NBR mínimo (Landsat) | nbr_min |
| 20 | `scar_size_range` | Tamanho das cicatrizes de fogo | scar_size_range |
| 21 | `severity` | Severidade potencial das áreas queimadas | fire_col5_severity |
| 22 | `time_after_fire` | Tempo desde a última ocorrência do fogo | fire_col5_time_after_fire |
| 23 | `unprecedented_fire` | Área queimada inédita | unprecedented_fire |
| 24 | `year_last_fire` | Última ocorrência do fogo | fire_col5_year_last_fire |


## Validação de configuração

Verifica a consistência dos arquivos YAML e batch antes de executar.


In [ ]:
# Validar configuracao YAML e batch
import os
!python -m src.mapbiomas_data.interfaces.cli --validate


In [ ]:
# ============================================================
# SETUP — roda uma vez no inicio da sessao
# ============================================================

import sys, os, subprocess

repo = "gif_factory"
if os.path.exists(repo):
    subprocess.run(["git", "-C", repo, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/wallyboy22/gif_factory.git"], check=True)
%cd gif_factory

# Limpar cache bytecode
!find . -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null; echo "Cache limpo"

# Garantir fontes
!apt-get install -qq fonts-dejavu-core 2>/dev/null; echo "OK"

!pip install -q earthengine-api pillow pyyaml google-cloud-storage

from google.colab import auth
auth.authenticate_user()

import ee
ee.Authenticate()
ee.Initialize(project="mapbiomas-fire-485203")

print("\nSetup concluido.")


### Grupo: Biomes (biomes)

**7 territórios × 24 produtos**

Sequência completa:
1. **Pipeline** — download EE → frames → collages → GIFs
2. **Area Stats** — (assíncrono) consulta EE, exporta área por classe para GCS
3. **Charts** — gera PNGs: distribuição anual + série temporal
4. **Compose** — funde mapas + gráficos lado a lado em GIF

Cada comando usa `--gcs` para salvar no storage permanente.
Use `--resume` para retomar se o ambiente cair.


In [ ]:
# (1) Gerar batch JSON para este grupo
import json, tempfile, os
collection_ds = "brasil_fire_col5"
group_id = "biomes"

from src.mapbiomas_data.config import ConfigLoader
cfg = ConfigLoader()
cfg.load_all()
ds = cfg.datasets.get(collection_ds, {})
product_ids = sorted(ds.get("products", {}).keys())

territory_ids = ["bioma_amazonia", "bioma_caatinga", "bioma_cerrado", "bioma_mata_atlantica", "bioma_pampa", "bioma_pantanal", "biomas_todos"]

items = []
for tid in territory_ids:
    for pid in product_ids:
        items.append({"dataset": collection_ds, "product": pid, "territory": tid})

n_terr = 7
n_prod = 24
print(f"Batch: {len(items)} combos ({group_id}: {n_terr} territorios x {n_prod} produtos)")

tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False)
json.dump({"items": items}, tmp)
tmp.close()
os.environ["OPTBATCH"] = tmp.name


In [ ]:
# (2) Pipeline: download EE → frames → collages → GIFs
# Pre-requisito: batch gerado, autenticacao EE OK
import os
!python -m src.mapbiomas_data.interfaces.cli --generate --batch $OPTBATCH --workers 6 --resume-from-gcs --font-scale 1.0 --gcs


In [ ]:
# (3) Area stats: computa area por classe (assincrono, EE task)
# Pre-requisito: pipeline executado (para config/territorios validos)
# Exporta CSV para GCS. Monitora tasks ate completar.
import os
!python -m src.mapbiomas_data.interfaces.cli --area-stats --batch $OPTBATCH --resume --gcs


In [ ]:
# (4) Charts: gera graficos anual + serie temporal
# Pre-requisito: area_stats CSVs disponiveis em area_stats/
# Gera PNGs em charts_annual/ e charts_timeseries/
import os
!python -m src.mapbiomas_data.interfaces.cli --charts --batch $OPTBATCH


In [ ]:
# (5) Compose: funde GIF do mapa + graficos lado a lado
# Pre-requisito: charts PNGs + GIFs gerados
# Gera GIF final em composed/
import os
!python -m src.mapbiomas_data.interfaces.cli --compose --batch $OPTBATCH


### Grupo: Country (country)

**2 territórios × 24 produtos**

Sequência completa:
1. **Pipeline** — download EE → frames → collages → GIFs
2. **Area Stats** — (assíncrono) consulta EE, exporta área por classe para GCS
3. **Charts** — gera PNGs: distribuição anual + série temporal
4. **Compose** — funde mapas + gráficos lado a lado em GIF

Cada comando usa `--gcs` para salvar no storage permanente.
Use `--resume` para retomar se o ambiente cair.


In [ ]:
# (1) Gerar batch JSON para este grupo
import json, tempfile, os
collection_ds = "brasil_fire_col5"
group_id = "country"

from src.mapbiomas_data.config import ConfigLoader
cfg = ConfigLoader()
cfg.load_all()
ds = cfg.datasets.get(collection_ds, {})
product_ids = sorted(ds.get("products", {}).keys())

territory_ids = ["pais_brasil", "pais_brasil_biomes"]

items = []
for tid in territory_ids:
    for pid in product_ids:
        items.append({"dataset": collection_ds, "product": pid, "territory": tid})

n_terr = 2
n_prod = 24
print(f"Batch: {len(items)} combos ({group_id}: {n_terr} territorios x {n_prod} produtos)")

tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False)
json.dump({"items": items}, tmp)
tmp.close()
os.environ["OPTBATCH"] = tmp.name


In [ ]:
# (2) Pipeline: download EE → frames → collages → GIFs
# Pre-requisito: batch gerado, autenticacao EE OK
import os
!python -m src.mapbiomas_data.interfaces.cli --generate --batch $OPTBATCH --workers 6 --resume-from-gcs --font-scale 1.0 --gcs


In [ ]:
# (3) Area stats: computa area por classe (assincrono, EE task)
# Pre-requisito: pipeline executado (para config/territorios validos)
# Exporta CSV para GCS. Monitora tasks ate completar.
import os
!python -m src.mapbiomas_data.interfaces.cli --area-stats --batch $OPTBATCH --resume --gcs


In [ ]:
# (4) Charts: gera graficos anual + serie temporal
# Pre-requisito: area_stats CSVs disponiveis em area_stats/
# Gera PNGs em charts_annual/ e charts_timeseries/
import os
!python -m src.mapbiomas_data.interfaces.cli --charts --batch $OPTBATCH


In [ ]:
# (5) Compose: funde GIF do mapa + graficos lado a lado
# Pre-requisito: charts PNGs + GIFs gerados
# Gera GIF final em composed/
import os
!python -m src.mapbiomas_data.interfaces.cli --compose --batch $OPTBATCH


### Grupo: Regions (regions)

**9 territórios × 24 produtos**

Sequência completa:
1. **Pipeline** — download EE → frames → collages → GIFs
2. **Area Stats** — (assíncrono) consulta EE, exporta área por classe para GCS
3. **Charts** — gera PNGs: distribuição anual + série temporal
4. **Compose** — funde mapas + gráficos lado a lado em GIF

Cada comando usa `--gcs` para salvar no storage permanente.
Use `--resume` para retomar se o ambiente cair.


In [ ]:
# (1) Gerar batch JSON para este grupo
import json, tempfile, os
collection_ds = "brasil_fire_col5"
group_id = "regions"

from src.mapbiomas_data.config import ConfigLoader
cfg = ConfigLoader()
cfg.load_all()
ds = cfg.datasets.get(collection_ds, {})
product_ids = sorted(ds.get("products", {}).keys())

territory_ids = ["regiao_bap", "regiao_bap_planalto", "regiao_centro_oeste", "regiao_matopiba", "regiao_matopiba_cerrado", "regiao_nordeste", "regiao_norte", "regiao_sudeste", "regiao_sul"]

items = []
for tid in territory_ids:
    for pid in product_ids:
        items.append({"dataset": collection_ds, "product": pid, "territory": tid})

n_terr = 9
n_prod = 24
print(f"Batch: {len(items)} combos ({group_id}: {n_terr} territorios x {n_prod} produtos)")

tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False)
json.dump({"items": items}, tmp)
tmp.close()
os.environ["OPTBATCH"] = tmp.name


In [ ]:
# (2) Pipeline: download EE → frames → collages → GIFs
# Pre-requisito: batch gerado, autenticacao EE OK
import os
!python -m src.mapbiomas_data.interfaces.cli --generate --batch $OPTBATCH --workers 6 --resume-from-gcs --font-scale 1.0 --gcs


In [ ]:
# (3) Area stats: computa area por classe (assincrono, EE task)
# Pre-requisito: pipeline executado (para config/territorios validos)
# Exporta CSV para GCS. Monitora tasks ate completar.
import os
!python -m src.mapbiomas_data.interfaces.cli --area-stats --batch $OPTBATCH --resume --gcs


In [ ]:
# (4) Charts: gera graficos anual + serie temporal
# Pre-requisito: area_stats CSVs disponiveis em area_stats/
# Gera PNGs em charts_annual/ e charts_timeseries/
import os
!python -m src.mapbiomas_data.interfaces.cli --charts --batch $OPTBATCH


In [ ]:
# (5) Compose: funde GIF do mapa + graficos lado a lado
# Pre-requisito: charts PNGs + GIFs gerados
# Gera GIF final em composed/
import os
!python -m src.mapbiomas_data.interfaces.cli --compose --batch $OPTBATCH


### Grupo: States (states)

**27 territórios × 24 produtos**

Sequência completa:
1. **Pipeline** — download EE → frames → collages → GIFs
2. **Area Stats** — (assíncrono) consulta EE, exporta área por classe para GCS
3. **Charts** — gera PNGs: distribuição anual + série temporal
4. **Compose** — funde mapas + gráficos lado a lado em GIF

Cada comando usa `--gcs` para salvar no storage permanente.
Use `--resume` para retomar se o ambiente cair.


In [ ]:
# (1) Gerar batch JSON para este grupo
import json, tempfile, os
collection_ds = "brasil_fire_col5"
group_id = "states"

from src.mapbiomas_data.config import ConfigLoader
cfg = ConfigLoader()
cfg.load_all()
ds = cfg.datasets.get(collection_ds, {})
product_ids = sorted(ds.get("products", {}).keys())

territory_ids = ["uf_acre", "uf_alagoas", "uf_amapa", "uf_amazonas", "uf_bahia", "uf_ceara", "uf_df", "uf_espirito_santo", "uf_goias", "uf_maranhao", "uf_mato_grosso", "uf_mato_grosso_do_sul", "uf_minas_gerais", "uf_para", "uf_paraiba", "uf_parana", "uf_pernambuco", "uf_piaui", "uf_rio_de_janeiro", "uf_rio_grande_do_norte", "uf_rio_grande_do_sul", "uf_rondonia", "uf_roraima", "uf_santa_catarina", "uf_sao_paulo", "uf_sergipe", "uf_tocantins"]

items = []
for tid in territory_ids:
    for pid in product_ids:
        items.append({"dataset": collection_ds, "product": pid, "territory": tid})

n_terr = 27
n_prod = 24
print(f"Batch: {len(items)} combos ({group_id}: {n_terr} territorios x {n_prod} produtos)")

tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False)
json.dump({"items": items}, tmp)
tmp.close()
os.environ["OPTBATCH"] = tmp.name


In [ ]:
# (2) Pipeline: download EE → frames → collages → GIFs
# Pre-requisito: batch gerado, autenticacao EE OK
import os
!python -m src.mapbiomas_data.interfaces.cli --generate --batch $OPTBATCH --workers 6 --resume-from-gcs --font-scale 1.0 --gcs


In [ ]:
# (3) Area stats: computa area por classe (assincrono, EE task)
# Pre-requisito: pipeline executado (para config/territorios validos)
# Exporta CSV para GCS. Monitora tasks ate completar.
import os
!python -m src.mapbiomas_data.interfaces.cli --area-stats --batch $OPTBATCH --resume --gcs


In [ ]:
# (4) Charts: gera graficos anual + serie temporal
# Pre-requisito: area_stats CSVs disponiveis em area_stats/
# Gera PNGs em charts_annual/ e charts_timeseries/
import os
!python -m src.mapbiomas_data.interfaces.cli --charts --batch $OPTBATCH


In [ ]:
# (5) Compose: funde GIF do mapa + graficos lado a lado
# Pre-requisito: charts PNGs + GIFs gerados
# Gera GIF final em composed/
import os
!python -m src.mapbiomas_data.interfaces.cli --compose --batch $OPTBATCH


In [ ]:
# Gerar batch para TODOS os grupos
import json, tempfile, os
collection_ds = "brasil_fire_col5"

from src.mapbiomas_data.config import ConfigLoader
cfg = ConfigLoader()
cfg.load_all()
ds = cfg.datasets.get(collection_ds, {})
product_ids = sorted(ds.get("products", {}).keys())

all_territories = ["bioma_amazonia", "bioma_caatinga", "bioma_cerrado", "bioma_mata_atlantica", "bioma_pampa", "bioma_pantanal", "biomas_todos", "pais_brasil", "pais_brasil_biomes", "regiao_bap", "regiao_bap_planalto", "regiao_centro_oeste", "regiao_matopiba", "regiao_matopiba_cerrado", "regiao_nordeste", "regiao_norte", "regiao_sudeste", "regiao_sul", "uf_acre", "uf_alagoas", "uf_amapa", "uf_amazonas", "uf_bahia", "uf_ceara", "uf_df", "uf_espirito_santo", "uf_goias", "uf_maranhao", "uf_mato_grosso", "uf_mato_grosso_do_sul", "uf_minas_gerais", "uf_para", "uf_paraiba", "uf_parana", "uf_pernambuco", "uf_piaui", "uf_rio_de_janeiro", "uf_rio_grande_do_norte", "uf_rio_grande_do_sul", "uf_rondonia", "uf_roraima", "uf_santa_catarina", "uf_sao_paulo", "uf_sergipe", "uf_tocantins"]

items = []
for tid in all_territories:
    for pid in product_ids:
        items.append({"dataset": collection_ds, "product": pid, "territory": tid})

n_terr = len(all_territories)
n_prod = len(product_ids)
print(f"Batch total: {len(items)} combos ({n_terr} territorios x {n_prod} produtos)")

tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False)
json.dump({"items": items}, tmp)
tmp.close()
os.environ["OPTBATCH"] = tmp.name


### Executar tudo (todos os grupos)

Sequência completa: pipeline → area stats → charts → compose.


In [ ]:
# Pipeline: todos os grupos
import os
!python -m src.mapbiomas_data.interfaces.cli --generate --batch $OPTBATCH --workers 6 --resume-from-gcs --font-scale 1.0 --gcs


In [ ]:
# Area stats: todos os grupos
import os
!python -m src.mapbiomas_data.interfaces.cli --area-stats --batch $OPTBATCH --resume --gcs


In [ ]:
# Charts: todos os grupos
import os
!python -m src.mapbiomas_data.interfaces.cli --charts --batch $OPTBATCH


In [ ]:
# Compose: todos os grupos
import os
!python -m src.mapbiomas_data.interfaces.cli --compose --batch $OPTBATCH


In [ ]:
# Sync final: index + Looker CSVs
# Os dados ja foram para o GCS via --gcs em cada etapa.
!python scripts/index/build_index.py --upload
!python scripts/looker/build_looker_csvs_from_gcs.py --dataset brasil_fire_col5


---
## Links úteis

| Recurso | Link |
|---|---|
| **Looker Studio** | [Abrir dashboard](https://datastudio.google.com/u/0/reporting/179f6b47-8f6e-4f51-abd5-75b7ae018a2b/page/XDzxF) |
| **GitHub** | [github.com/wallyboy22/gif_factory](https://github.com/wallyboy22/gif_factory) |
| **MapBiomas** | [plataforma.brasil.mapbiomas.org](https://plataforma.brasil.mapbiomas.org) |

---

*Gerado por [Fábrica de GIFs — IPAM / MapBiomas](https://github.com/wallyboy22/gif_factory)*
